In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [2]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

aurora_gate_expense_categorization_challenge_path = kagglehub.competition_download('aurora-gate-expense-categorization-challenge')

print('Data source import complete.')


Data source import complete.


In [3]:
print('Installing catboost...')
!pip install catboost
print('Catboost installed successfully.')

Installing catboost...
Catboost installed successfully.


### Fixing `FileNotFoundError` for `train.csv` and `test.csv`

I've modified the `discover_data_paths` function to explicitly check for the data downloaded by `kagglehub.competition_download` at the path stored in `aurora_gate_expense_categorization_challenge_path`. This ensures the pipeline can correctly locate and load the necessary training and testing data.


In [4]:
# =============================================================================
# 1. DEPENDENCIES & ENVIRONMENT SETUP
# =============================================================================
import os
import sys
import json
import logging
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional, Iterable

import numpy as np
import pandas as pd
import joblib
import lightgbm
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, TimeSeriesSplit
from sklearn.utils.class_weight import compute_class_weight

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
print('✅ Environment & Dependencies Successfully Loaded!', flush=True)


✅ Environment & Dependencies Successfully Loaded!


# 🏆 AuroraGate — Expense Categorization Challenge (Phase 2 High-Precision Ensemble)
> **Authors:** Ali Azizi Deh Sorkh & Antigravity (Google DeepMind)
> **Architecture:** Deep CatBoost + LightGBM Ensemble with 1500 TF-IDF Character n-grams, Weight Blending Grid Search, & 100,000 Random Search Threshold Tuning
> **Metric:** Macro F1-Score (10 Target Expense Categories)

---
## 📌 Overview & Pipeline Strategy
This self-contained Kaggle notebook implements the complete **Phase 2 High-Precision Pipeline**:
1. **Data Ingestion & Path Detection**: Auto-detects Kaggle Dataset inputs (`/kaggle/input/`) or local data (`./data`).
2. **Feature Engineering**: Character-level TF-IDF (1500 features), store name regex parsing, temporal encodings, log amount percentiles, recurring pattern flags, and transaction gap sequences.
3. **Dual Gradient Boosted Ensemble**: Stratified 5-Fold CV training of **CatBoostClassifier** and **LGBMClassifier** with class-weighted objective.
4. **Weight Blending Grid Search**: Optimizes blending alpha ($lpha \cdot \text{CatBoost} + (1-\alpha) \cdot \text{LightGBM}$) across 41 evaluation steps.
5. **100,000 Random Search Threshold Tuning**: Optimizes per-class probability multipliers to directly maximize Macro F1 score on Out-Of-Fold predictions.
6. **Final Model Export & Submission**: Saves trained models, vectorizer, thresholds, and generates `submission.csv`.


In [5]:
# =============================================================================
# 2. KAGGLE DATA PATH DISCOVERY (RECURSIVE RGLOB)
# =============================================================================
def discover_data_paths() -> Tuple[Path, Path]:
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        train_matches = list(kaggle_input.rglob('train.csv'))
        if train_matches:
            found_dir = train_matches[0].parent
            print(f'📍 Found Kaggle Input Path (rglob, flush=True): {found_dir}', flush=True)
            return found_dir, Path('/kaggle/working')

    for candidate in [Path('./data'), Path('../data'), Path('../../data')]:
        if candidate.exists() and (candidate / 'train.csv').exists():
            print(f'📍 Found Local Data Path: {candidate.resolve()}', flush=True)
            return candidate.resolve(), Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.').resolve()

    local_data = Path('./data')
    working_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.').resolve()
    return local_data.resolve(), working_dir

DATA_DIR, OUTPUT_DIR = discover_data_paths()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = OUTPUT_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TARGET_COLUMN = 'category'
TARGET_CATEGORIES = [
    'Food & Dining', 'Groceries', 'Transportation', 'Entertainment', 'Shopping',
    'Bills & Utilities', 'Health & Fitness', 'Miscellaneous', 'Subscriptions', 'Travel'
]


📍 Found Kaggle Input Path (rglob, flush=True): /kaggle/input/competitions/aurora-gate-expense-categorization-challenge


In [6]:
# =============================================================================
# 2. KAGGLE DATA PATH DISCOVERY (RECURSIVE RGLOB)
# =============================================================================
def discover_data_paths() -> Tuple[Path, Path]:
    # Check for the downloaded Kaggle competition data specifically
    # The aurora_gate_expense_categorization_challenge_path is set in cell M9-8MEiBzpdA
    # and should be available as a global variable after that cell runs.
    if 'aurora_gate_expense_categorization_challenge_path' in globals():
        downloaded_path = Path(globals()['aurora_gate_expense_categorization_challenge_path'])
        if downloaded_path.exists():
            train_file = downloaded_path / 'train.csv'
            test_file = downloaded_path / 'test.csv'
            if train_file.exists() and test_file.exists():
                print(f'📍 Found Kagglehub Download Path: {downloaded_path}', flush=True)
                return downloaded_path, Path('/kaggle/working')

    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        train_matches = list(kaggle_input.rglob('train.csv'))
        if train_matches:
            found_dir = train_matches[0].parent
            print(f'📍 Found Kaggle Input Path (rglob, flush=True): {found_dir}', flush=True)
            return found_dir, Path('/kaggle/working')

    for candidate in [Path('./data'), Path('../data'), Path('../../data')]:
        if candidate.exists() and (candidate / 'train.csv').exists():
            print(f'📍 Found Local Data Path: {candidate.resolve()}', flush=True)
            return candidate.resolve(), Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.').resolve()

    local_data = Path('./data')
    working_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.').resolve()
    return local_data.resolve(), working_dir

DATA_DIR, OUTPUT_DIR = discover_data_paths()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = OUTPUT_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TARGET_COLUMN = 'category'
TARGET_CATEGORIES = [
    'Food & Dining', 'Groceries', 'Transportation', 'Entertainment', 'Shopping',
    'Bills & Utilities', 'Health & Fitness', 'Miscellaneous', 'Subscriptions', 'Travel'
]


📍 Found Kagglehub Download Path: /kaggle/input/competitions/aurora-gate-expense-categorization-challenge


In [7]:
# =============================================================================
# 3. FEATURE ENGINEERING PIPELINE (Source Patch)
# =============================================================================
def extract_store_name(description: str) -> str:
    if not description or pd.isna(description):
        return 'UNKNOWN'
    text = str(description).upper().strip()
    for kw in ['POS PURCHASE', 'DEBIT CARD PURCHASE', 'CARD PAYMENT', 'WM SUPERCENTER']:
        text = text.replace(kw, '').strip()
    tokens = text.split()
    return tokens[0] if tokens else 'UNKNOWN'

def engineer_features(df: pd.DataFrame, is_train: bool = True) -> pd.DataFrame:
    features = df.copy()
    descriptions = features['description'].fillna('').astype(str)
    dates = pd.to_datetime(features['date'], errors='coerce')
    amounts = pd.to_numeric(features['amount'], errors='coerce')

    features['store_name'] = descriptions.apply(extract_store_name)
    features['is_round_amount'] = amounts.mod(1).fillna(0).eq(0).astype(int)
    features['is_round_5'] = amounts.mod(5).fillna(0).eq(0).astype(int)
    features['is_round_10'] = amounts.mod(10).fillna(0).eq(0).astype(int)
    features['amount_percentile'] = amounts.rank(pct=True).fillna(0)
    day_of_year = dates.dt.dayofyear.fillna(0).astype(int)
    month = dates.dt.month.fillna(0).astype(int)
    features['day_of_year'] = day_of_year
    features['month'] = month
    features['month_sin'] = np.sin(2 * np.pi * month / 12.0)
    features['month_cos'] = np.cos(2 * np.pi * month / 12.0)
    features['day_of_year_sin'] = np.sin(2 * np.pi * day_of_year / 365.25)
    features['day_of_year_cos'] = np.cos(2 * np.pi * day_of_year / 365.25)
    features['week_of_year'] = dates.dt.isocalendar().week.fillna(0).astype(int)
    features['days_to_weekend'] = (5 - dates.dt.dayofweek).clip(lower=0).fillna(0).astype(int)
    features['is_weekend'] = dates.dt.dayofweek.ge(5).fillna(False).astype(int)
    features['quarter'] = dates.dt.quarter.fillna(0).astype(int)
    features['day_of_month'] = dates.dt.day.fillna(0).astype(int)
    features['month_period'] = pd.cut(dates.dt.day, bins=[0, 10, 20, 31], labels=['start', 'mid', 'end']).astype(str).fillna('unknown')
    features['amount_bins'] = pd.cut(amounts, bins=[-np.inf, 25, 100, np.inf], labels=['small', 'medium', 'large']).astype(str).fillna('unknown')
    features['log_amount'] = np.log1p(amounts.clip(lower=0))

    features['description_word_count'] = descriptions.str.split().str.len().fillna(0).astype(int)
    features['description_char_count'] = descriptions.str.len().fillna(0).astype(int)
    features['description_digit_count'] = descriptions.str.count(r'\d').fillna(0).astype(int)

    upper = descriptions.str.upper()
    features['has_food'] = upper.str.contains(r'RESTAURANT|CAFE|COFFEE|BURGER|PIZZA|FOOD', regex=True).astype(int)
    features['has_grocery'] = upper.str.contains(r'MART|SUPERMARKET|GROCERY|WHOLEFOODS|WALMART', regex=True).astype(int)
    features['has_uber'] = upper.str.contains(r'UBER|LYFT|TAXI|TRANSIT|CAB', regex=True).astype(int)
    features['has_health'] = upper.str.contains(r'PHARMACY|HEALTH|CLINIC|DOCTOR|FITNESS|GYM', regex=True).astype(int)
    features['has_bill'] = upper.str.contains(r'UTILITY|ELECTRIC|WATER|GAS|BILL|INTERNET|MOBILE', regex=True).astype(int)

    return features

def categorical_feature_names() -> List[str]:
    return ['day_of_week', 'month', 'quarter', 'month_period', 'amount_bins', 'store_name']


In [8]:
# =============================================================================
# 4. MODEL FRAME PREPARATION & 1500 TF-IDF VECTORIZATION
# =============================================================================
def prepare_model_frames(
    engineered_df: pd.DataFrame,
    vectorizer: Optional[TfidfVectorizer] = None,
    is_train: bool = True
) -> Tuple[pd.DataFrame, pd.DataFrame, List[str], List[str], TfidfVectorizer]:
    categorical = [col for col in categorical_feature_names() if col in engineered_df.columns]
    text_columns = ['description']
    excluded = {'date', 'transaction_id', TARGET_COLUMN}
    base_cols = [col for col in engineered_df.columns if col not in excluded]

    df = engineered_df[base_cols].copy()
    descriptions = df['description'].fillna('').astype(str)

    if is_train or vectorizer is None:
        vectorizer = TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=(3, 5),
            max_features=1500,
            sublinear_tf=True,
        )
        tfidf_matrix = vectorizer.fit_transform(descriptions).toarray().astype(np.float32)
    else:
        tfidf_matrix = vectorizer.transform(descriptions).toarray().astype(np.float32)

    tfidf_cols = [f'tfidf_{i}' for i in range(tfidf_matrix.shape[1])]
    tfidf_df = pd.DataFrame(tfidf_matrix, columns=tfidf_cols, index=df.index)
    full_frame = pd.concat([df, tfidf_df], axis=1)

    catboost_frame = full_frame.copy()
    catboost_frame['description'] = catboost_frame['description'].fillna('unknown').astype(str)
    lgbm_frame = full_frame.copy()

    for col in categorical:
        catboost_frame[col] = catboost_frame[col].fillna('unknown').astype(str)
        lgbm_frame[col] = lgbm_frame[col].fillna('unknown').astype('category')

    lgbm_frame = lgbm_frame.drop(columns=text_columns, errors='ignore')
    return catboost_frame, lgbm_frame, categorical, text_columns, vectorizer


In [9]:
# =============================================================================
# 5. ENSEMBLE WEIGHT BLENDING & 100K THRESHOLD OPTIMIZER
# =============================================================================
def augment_minority_classes(df: pd.DataFrame, target_cols: List[str] = ['Travel', 'Subscriptions'], factor: float = 1.5, random_state: int = 42) -> pd.DataFrame:
    rng = np.random.RandomState(random_state)
    augmented_rows = []
    for cat in target_cols:
        cat_df = df[df['category'] == cat]
        n_copies = int(len(cat_df) * (factor - 1.0))
        if n_copies > 0:
            sample_df = cat_df.sample(n_copies, replace=True, random_state=random_state).copy()
            sample_df['amount'] = (sample_df['amount'] * rng.uniform(0.98, 1.02, size=len(sample_df))).round(2)
            augmented_rows.append(sample_df)
    if augmented_rows:
        return pd.concat([df] + augmented_rows, axis=0).reset_index(drop=True)
    return df

def apply_domain_constraints(probas: np.ndarray, amounts: np.ndarray, classes: np.ndarray) -> np.ndarray:
    bounds = {
        'Bills & Utilities': (30.0, 265.0),
        'Entertainment': (0.0, 76.0),
        'Food & Dining': (0.0, 66.0),
        'Groceries': (0.0, 222.0),
        'Health & Fitness': (0.0, 182.0),
        'Miscellaneous': (0.0, 510.0),
        'Shopping': (0.0, 355.0),
        'Subscriptions': (0.0, 56.0),
        'Transportation': (0.0, 91.0),
        'Travel': (60.0, 910.0)
    }
    constrained = probas.copy()
    for idx, cat_name in enumerate(classes):
        if cat_name in bounds:
            min_a, max_a = bounds[cat_name]
            mask = (amounts < min_a) | (amounts > max_a)
            constrained[mask, idx] = 0.0
    return constrained

def optimize_ensemble_weight(
    y_true: np.ndarray,
    oof_catboost: np.ndarray,
    oof_lgbm: np.ndarray,
    classes: np.ndarray
) -> Tuple[float, float]:
    best_alpha = 0.5
    best_score = -1.0
    for alpha in np.linspace(0.0, 1.0, 41):
        blend = alpha * oof_catboost + (1.0 - alpha) * oof_lgbm
        preds = classes[np.argmax(blend, axis=1)]
        score = f1_score(y_true, preds, average='macro', zero_division=0)
        if score > best_score:
            best_score = score
            best_alpha = float(alpha)
    return best_alpha, best_score

def optimize_class_thresholds_100k(
    y_true: np.ndarray,
    oof_probabilities: np.ndarray,
    classes: np.ndarray,
    n_samples: int = 100000
) -> Dict[str, float]:
    num_classes = len(classes)
    best_multipliers = np.ones(num_classes)
    scaled_base = oof_probabilities * best_multipliers
    preds_base = classes[np.argmax(scaled_base, axis=1)]
    best_score = f1_score(y_true, preds_base, average='macro', zero_division=0)

    rng = np.random.RandomState(RANDOM_STATE)
    batch_size = 5000
    n_batches = n_samples // batch_size
    for _ in range(n_batches):
        candidates = rng.uniform(0.4, 1.6, size=(batch_size, num_classes))
        for candidate in candidates:
            scaled = oof_probabilities * candidate
            preds = classes[np.argmax(scaled, axis=1)]
            score = f1_score(y_true, preds, average='macro', zero_division=0)
            if score > best_score:
                best_score = score
                best_multipliers = candidate
    return {str(cls): float(best_multipliers[i]) for i, cls in enumerate(classes)}

def apply_class_thresholds(
    probabilities: np.ndarray,
    classes: np.ndarray,
    thresholds: Dict[str, float]
) -> np.ndarray:
    multipliers = np.array([thresholds.get(str(cls), 1.0) for cls in classes])
    scaled_probas = probabilities * multipliers
    return classes[np.argmax(scaled_probas, axis=1)]
